# 🎨 04 — Visualise Results
### Amazon Deforestation Change Detection Pipeline

Produces 7 publication-quality figures from the change detection outputs.

---
**Pipeline:**
1. ✅ `01_download_data.ipynb`
2. ✅ `02_prepare_scenes.ipynb`
3. ✅ `03_detect_changes.ipynb`
4. 🎨 `04_visualise.ipynb` ← *you are here*


## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import rasterio
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

PREPARED_DIR = Path("data/prepared")
CHANGE_DIR   = Path("outputs/change")
OUTPUT_DIR   = Path("outputs/change")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Annual Crop", "Forest", "Herbaceous Veg", "Highway",
    "Industrial", "Pasture", "Permanent Crop", "Residential",
    "River", "Sea / Lake",
]
CLASS_ICONS  = ["🌾","🌲","🌿","🛣️","🏭","🐄","🍇","🏘️","🏞️","🌊"]
CLASS_COLORS = [
    (0.78,0.52,0.25),(0.13,0.37,0.13),(0.42,0.68,0.35),(0.65,0.60,0.45),
    (0.45,0.45,0.55),(0.55,0.75,0.40),(0.25,0.55,0.15),(0.75,0.60,0.50),
    (0.20,0.50,0.80),(0.10,0.30,0.70),
]
PIXEL_SIZE_M       = 10.0
HECTARES_PER_PIXEL = (PIXEL_SIZE_M ** 2) / 10_000

def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(), src.profile

def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3))
    for i, c in enumerate(CLASS_COLORS):
        rgb[mask == i] = c
    return rgb

def legend_handles():
    return [mpatches.Patch(color=c, label=f"{CLASS_ICONS[i]} {CLASS_NAMES[i]}")
            for i, c in enumerate(CLASS_COLORS)]

def save(fig, name):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"✅ Saved: {path}\n")

print("✅ Setup complete")


## 🛰️ Figure 1 — RGB Comparison: Before & After

In [ ]:
rgb_2019 = np.transpose(load_raster(PREPARED_DIR/"2019_rgb.tif")[0], (1,2,0)).clip(0,1)
rgb_2024 = np.transpose(load_raster(PREPARED_DIR/"2024_rgb.tif")[0], (1,2,0)).clip(0,1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Amazon — Sentinel-2 RGB: Before & After", fontsize=14, fontweight="bold")
for ax, img, year, col in zip(axes, [rgb_2019, rgb_2024], ["2019","2024"], ["#2ecc71","#e74c3c"]):
    ax.imshow(img); ax.set_title(f"🛰️ {year}", fontsize=13, fontweight="bold", color=col)
    ax.axis("off")
    for s in ax.spines.values():
        s.set_edgecolor(col); s.set_linewidth(3); s.set_visible(True)
plt.tight_layout()
save(fig, "01_rgb_comparison.png")


## 🗺️ Figure 2 — DOFA Segmentation Maps

In [ ]:
seg_2019 = load_raster(CHANGE_DIR/"segmentation_2019.tif")[0][0]
seg_2024 = load_raster(CHANGE_DIR/"segmentation_2024.tif")[0][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Amazon — DOFA Land Cover Segmentation: 2019 vs 2024", fontsize=14, fontweight="bold")
for ax, seg, year in zip(axes, [seg_2019, seg_2024], ["2019","2024"]):
    ax.imshow(mask_to_rgb(seg), interpolation="nearest")
    ax.set_title(f"Land Cover {year}", fontsize=12, fontweight="bold"); ax.axis("off")
fig.legend(handles=legend_handles(), loc="lower center", ncol=5,
           bbox_to_anchor=(0.5,-0.06), fontsize=9, framealpha=0.9)
plt.tight_layout()
save(fig, "02_segmentation_comparison.png")


## 🔴 Figure 3 — Binary Change Map

In [ ]:
binary  = load_raster(CHANGE_DIR/"binary_change.tif")[0][0]
ha      = binary.sum() * HECTARES_PER_PIXEL
pct     = binary.sum() / binary.size * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Amazon — Binary Change Detection: 2019 → 2024", fontsize=14, fontweight="bold")
axes[0].imshow(rgb_2019); axes[0].set_title("🛰️ 2019 Reference"); axes[0].axis("off")
axes[1].imshow(rgb_2019)
axes[1].imshow(binary, cmap=LinearSegmentedColormap.from_list("c",[(0,0,0,0),(0.9,0.1,0.1,0.7)]),
               vmin=0, vmax=1)
axes[1].set_title(f"🔴 Changed
{ha:,.0f} ha  ({pct:.1f}%)", fontsize=11, color="#e74c3c")
axes[1].axis("off")
axes[2].imshow(binary, cmap="Reds", vmin=0, vmax=1, interpolation="nearest")
axes[2].set_title("Change Heatmap"); axes[2].axis("off")
plt.tight_layout()
save(fig, "03_binary_change.png")


## 🌈 Figure 4 — Class-Level Change Map

In [ ]:
changed = binary.astype(bool)
H, W    = seg_2019.shape
change_rgb = np.ones((H, W, 3)) * 0.15
for i, c in enumerate(CLASS_COLORS):
    change_rgb[changed & (seg_2024 == i)] = c

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Amazon — What Did the Land Become?", fontsize=14, fontweight="bold")
axes[0].imshow(mask_to_rgb(seg_2019), interpolation="nearest")
axes[0].set_title("Land Cover 2019 (Before)"); axes[0].axis("off")
axes[1].imshow(change_rgb, interpolation="nearest")
axes[1].set_title("Changed Pixels — Coloured by 2024 Class\n(Dark = Unchanged)"); axes[1].axis("off")
fig.legend(handles=legend_handles(), loc="lower center", ncol=5,
           bbox_to_anchor=(0.5,-0.06), fontsize=9, framealpha=0.9)
plt.tight_layout()
save(fig, "04_class_change_map.png")


## 🌿 Figure 5 — NDVI Comparison

In [ ]:
n19  = load_raster(PREPARED_DIR/"2019_ndvi.tif")[0][0]
n24  = load_raster(PREPARED_DIR/"2024_ndvi.tif")[0][0]
diff = n24 - n19

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("NDVI (Vegetation Index): 2019 vs 2024\nNegative change = forest loss",
             fontsize=13, fontweight="bold")
for ax, data, title in zip(axes[:2], [n19, n24], [f"NDVI 2019\nMean={n19.mean():.3f}",
                                                   f"NDVI 2024\nMean={n24.mean():.3f}"]):
    im = ax.imshow(data, cmap="RdYlGn", vmin=-0.2, vmax=1.0)
    ax.set_title(title); ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.046)
im = axes[2].imshow(diff, cmap="RdBu", vmin=-0.5, vmax=0.5)
axes[2].set_title(f"Change (2024-2019)\nMean={diff.mean():.3f}",
                  color="#c0392b" if diff.mean()<0 else "#27ae60")
axes[2].axis("off"); plt.colorbar(im, ax=axes[2], fraction=0.046, label="NDVI change")
plt.tight_layout()
save(fig, "05_ndvi_comparison.png")


## 📊 Figure 6 — Top Land Cover Transitions

In [ ]:
csv_path = CHANGE_DIR / "change_summary.csv"
df = pd.read_csv(csv_path).head(12) if csv_path.exists() else pd.DataFrame()

if not df.empty:
    labels     = [f"{r['from_class']} → {r['to_class']}" for _, r in df.iterrows()]
    bar_colors = [CLASS_COLORS[CLASS_NAMES.index(r["to_class"])] if r["to_class"] in CLASS_NAMES
                  else CLASS_COLORS[0] for _, r in df.iterrows()]
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(labels, df["hectares"], color=bar_colors, edgecolor="white")
    ax.bar_label(bars, fmt=lambda x: f"{x:,.0f} ha", padding=4, fontsize=9)
    ax.set_xlabel("Area (hectares)"); ax.invert_yaxis()
    ax.set_title("Top Land Cover Transitions — 2019 to 2024\n(Colour = destination class)",
                 fontsize=13, fontweight="bold")
    ax.set_facecolor("#f8f9fa"); fig.patch.set_facecolor("white")
    plt.tight_layout()
    save(fig, "06_transition_chart.png")
else:
    print("⚠️ change_summary.csv not found — run 03_detect_changes.ipynb first")


## 📋 Figure 7 — Summary Card

In [ ]:
forest_cls      = 1
forest_2019_ha  = (seg_2019 == forest_cls).sum() * HECTARES_PER_PIXEL
forest_2024_ha  = (seg_2024 == forest_cls).sum() * HECTARES_PER_PIXEL
forest_lost_ha  = max(0, forest_2019_ha - forest_2024_ha)
pct_lost        = forest_lost_ha / max(forest_2019_ha, 1) * 100
total_ha        = binary.sum() * HECTARES_PER_PIXEL
ndvi_change     = n24.mean() - n19.mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off"); fig.patch.set_facecolor("#1a1a2e")
ax.text(0.5, 0.95, "🌳 Amazon Deforestation — Change Detection Summary",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=14, fontweight="bold", color="white")

stats = [
    ("Total Area Changed",  f"{total_ha:,.0f} ha",                         "#e74c3c"),
    ("Forest Cover 2019",   f"{forest_2019_ha:,.0f} ha",                   "#2ecc71"),
    ("Forest Cover 2024",   f"{forest_2024_ha:,.0f} ha",                   "#e67e22"),
    ("Forest Lost",         f"{forest_lost_ha:,.0f} ha  ({pct_lost:.1f}%)","#c0392b"),
    ("NDVI Change",         f"{ndvi_change:+.3f}",                          "#e74c3c" if ndvi_change<0 else "#2ecc71"),
    ("Model",               "DOFA Foundation Model",                        "#3498db"),
    ("Data Source",         "Sentinel-2 L2A (ESA Copernicus)",              "#9b59b6"),
    ("Period",              "2019 → 2024",                                  "white"),
]
for i, (label, value, color) in enumerate(stats):
    y = 0.80 - i * 0.10
    ax.text(0.05, y, f"{label}:", transform=ax.transAxes, ha="left",
            fontsize=11, color="#aaaaaa")
    ax.text(0.50, y, value, transform=ax.transAxes, ha="left",
            fontsize=11, fontweight="bold", color=color)
ax.text(0.5, 0.02, "Built with DOFA · Sentinel-2 · geo-deep-learning · PyTorch",
        transform=ax.transAxes, ha="center", fontsize=9, color="#666666", style="italic")
save(fig, "07_summary_card.png")

print("\n🎉 All visualisations complete!")
print(f"   Saved to: {OUTPUT_DIR.resolve()}")
